In [28]:
import torch
from torch_geometric.datasets import BA2MotifDataset
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
import pickle

In [2]:
dataset_root="./data/BA2Motif"
dataset = BA2MotifDataset(root=dataset_root)

Processing...
Done!


In [3]:
print(f"lenght of dataset : {len(dataset)}")
print(f"number of class : {dataset.num_classes}")
print(f"number of features : {dataset.num_features}")


lenght of dataset : 1000
number of class : 2
number of features : 10


In [7]:
data=dataset[0]
print(" construction of first graph :")
print(f" edge matrix : {data.edge_index.shape}")
print(f" nodes features : {data.x.shape}")
print(f" label : {data.y.item()}")


 construction of first graph :
 edge matrix : torch.Size([2, 50])
 nodes features : torch.Size([25, 10])
 label : 0


In [ ]:
num_samples = 3
for idx in range(min(num_samples, len(dataset))):
    data = dataset[idx]
    print(f"\n{'='*50}")
    print(f"Sample {idx}:")
    print(f"{'='*50}")
    
    print(f"1. Node features (x): shape = {data.x.shape}")
    print(f"   First 2 rows (if enough nodes):\n{data.x[:2]}")  
    
    print(f"2. Edge index (edge_index): shape = {data.edge_index.shape}")
    print(f"   Number of edges: {data.edge_index.size(1)}")
    print(f"   First 5 edges (source -> target):\n{data.edge_index[:, :5].t()}")  
    
    print(f"3. Graph label (y): {data.y.item()} -> {'Positive (House motif)' if data.y.item() == 1 else 'Negative (Cycle motif)'}")
    
    if hasattr(data, 'motif_mask'):
        motif_mask = data.motif_mask
        print(f"4. Motif mask (motif_mask): shape = {motif_mask.shape}")
        num_motif_nodes = motif_mask.sum().item()
        print(f"   Number of nodes in motif: {num_motif_nodes}")
        print(f"   Indices of motif nodes: {motif_mask.nonzero(as_tuple=True)[0].tolist()}")
    elif hasattr(data, 'ground_truth_mask'):
        print(f"4. Ground truth mask: shape = {data.ground_truth_mask.shape}")
    else:
        print("4. No motif mask found in this dataset sample.")

    exclude_attrs = ['stores','node_stores','edge_stores', '__dict__', '__module__', '__weakref__']

    print("\nAll attributes of this data object:")
    for attr in dir(data):
        if attr.startswith('_') or attr in exclude_attrs:
            continue
        val = getattr(data, attr)
        if not callable(val):
            if isinstance(val, torch.Tensor):
                print(f"   - {attr}: tensor of shape {val.shape}")
            else:
                print(f"   - {attr}: {val}")


Sample 0:
1. Node features (x): shape = torch.Size([25, 10])
   First 2 rows (if enough nodes):
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
         0.1000],
        [0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
         0.1000]])
2. Edge index (edge_index): shape = torch.Size([2, 50])
   Number of edges: 50
   First 5 edges (source -> target):
tensor([[ 0,  1],
        [ 0,  3],
        [ 0,  4],
        [ 0,  7],
        [ 0, 13]])
3. Graph label (y): 0 -> Negative (Cycle motif)
4. No motif mask found in this dataset sample.

All attributes of this data object:
   - batch: None
   - edge_attr: None
   - edge_index: tensor of shape torch.Size([2, 50])
   - edge_weight: None
   - face: None
   - is_cuda: False
   - node_offsets: {None: 0}
   - num_edge_features: 0
   - num_edge_types: 1
   - num_edges: 50
   - num_faces: None
   - num_features: 10
   - num_node_features: 10
   - num_node_types: 1
   - num_nodes: 25
   - p

C:\Users\mahboub\AppData\Local\Temp\ipykernel_11396\3306521073.py:34: UserWarning: 'num_faces' is deprecated, use 'data.face.size(-1)' instead
  val = getattr(data, attr)


In [24]:
total_len=len(dataset)
train_len=int(0.8*total_len)
test_len=int(0.1*total_len)
val_len=total_len-train_len-test_len
print(train_len,test_len,val_len)


800 100 100


In [27]:
train_dataset,val_dataset,test_dataset=random_split(
    dataset,[train_len,val_len,test_len]
    ,generator=torch.Generator().manual_seed(42))

In [29]:
train_indices=train_dataset.indices
val_indices=val_dataset.indices
test_indices=test_dataset.indices

splits={
    'train':train_indices,
    'val':val_indices,
    'test':test_indices
}

with open('./data/splits.pkl','wb') as f:
    pickle.dump(splits,f)

print("train,test,validation indices saved in splits.pkl")

train,test,validation indices saved in splits.pkl
